In [2]:
import gc
import math
import os
import sys
import warnings

import numpy as np
import optuna
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

name = "gdsc1"
# task = "cell"
task = "drug"


method = "GAT"

target_dim = [
    1,  # Drug
    # 0  # Cell
]

PATH = f"../{name}_data/"

In [3]:
(
    res,
    pos_num,
    null_mask,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

load gdsc1
unique drugs: 73
unique genes: 166
DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099
Final gene exp shape: (916, 2099)
Final drug Act shape: (331, 916)


100%|██████████| 25/25 [00:03<00:00,  8.23it/s]


Done!


In [32]:

n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        p = res.iloc[:, target_index].dropna() > 0
        tmp = sum(p)*100/len(p)
        if 0 < tmp < 100:
            if dim:
                if drug_sum[target_index] < 10:
                    continue
            else:
                if cell_sum[target_index] < 10:
                    continue

            for fold in range(n_kfold):
                sampler = NewSampler(
                    res,
                    null_mask.T.values,
                    target_dim,
                    target_index,
                    S_d,
                    S_c,
                    S_g,
                    A_cg,
                    A_dg,
                    PATH,
                    seed,
                )
            else:
                print(f"Target {target_index} skipped: All labels are {'0' if tmp == 0 else '1'}.")

331it [00:00, 6877.03it/s]

20.85
2.07
30.9
5.9
6.0
1.53
5.46
59.28
85.7
3.82
8.08
32.53
21.83
6.99
3.6
6.77
54.91
2.62
5.46
91.48
17.9
12.99
5.13
4.04
6.88
62.12
64.96
47.49
7.86
2.18
9.06
21.18
5.35
1.09
17.9
3.28
45.2
3.6
4.91
36.14
34.93
34.93
4.04
3.71
7.21
26.86
7.97
2.95
46.94
22.71
14.19
13.97
41.05
8.95
89.85
3.6
29.15
93.67
42.25
37.88
4.69
6.66
1.64
1.64
58.95
8.08
2.62
5.13
5.02
1.64
42.36
6.0
87.55
91.38
68.12
14.74
96.94
77.18
84.83
90.07
11.9
2.4
89.3
1.64
22.27
2.4
1.2
6.11
80.68
43.56
49.02
13.97
6.77
3.6
1.2
4.69
2.07
36.14
17.9
69.54
94.76
1.86
26.2
12.66
2.73
42.36
47.16
1.2
78.71
20.63
15.5
1.42
1.86
9.28
1.42
63.43
2.51
7.97
83.52
23.25
10.37
1.42
51.75
49.78
61.35
6.33
86.46
4.48
12.55
21.62
1.86
33.41
3.17
64.08
88.32
2.84
61.9
2.4
5.35
1.53
13.21
36.14
7.64
2.07
1.31
37.23
12.55
90.83
1.42
16.05
8.41
34.39
19.87
23.8
32.1
18.56
1.86
22.05
2.84
33.73
16.92
13.97
10.04
1.86
94.0
96.62
58.19
2.4
8.08
16.16
81.99
46.4
1.2
1.75
5.02
70.52
2.29
10.81
59.39
86.46
1.09
83.41
13.97
24.13
67.69
10.

In [49]:
for i in range(len(res.columns)):
    p = res.iloc[:, i].dropna() > 0
    tmp = (round(sum(p)*100/len(p), 2))
    if (tmp == 0) or (tmp == 100):
        print(i)

4
7
47
49
57
75
79
91
100
102
112
113
115
122
123
125
137
152
156
161
165
166
167
168
169
172
182
187
192
210
213
228
242
249
250
255
259
261
263
264
271
274
283
294
296
298
301
304
306
315
319
327
328


In [48]:
for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
    p = res.iloc[:, target_index].dropna() > 0
    tmp = (round(sum(p)*100/len(p), 2))
    if (tmp == 0) or (tmp == 100):
        print(target_index)

331it [00:00, 5261.68it/s]

4
7
47
49
57
75
79
91
100
102
112
113
115
122
123
125
137
152
156
161
165
166
167
168
169
172
182
187
192
210
213
228
242
249
250
255
259
261
263
264
271
274
283
294
296
298
301
304
306
315
319
327
328


In [22]:
res

,(5Z)-7-Oxozeaenol,5-Fluorouracil,A-443654,A-770041,A-83-01,ACY-1215,AGI-6780,AICA Ribonucleotide,AKT inhibitor VIII_171,AKT inhibitor VIII_228,...,YK-4-279,YM201636,Z-LLNle-CHO,ZG-10,ZM447439,ZSTK474,Zibotentan,"eEF2K Inhibitor, A-484954",kb NB 142-70,torin2
22RV1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
23132-87,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
42-MG-BA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
451Lu,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5637,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
ZR-75-30,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
huH-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
no-10,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
